# Chapter 8 gallery — robust ARIMA (`arima_rob`)

Reproduces the six Chapter-8 time-series scripts via `rpm.arima_rob` → `robustarima::arima.rob` (the *filtered tau-estimate*):

| Script | Example | Call |
|---|---|---|
| `resex.R` | 8.6 | `arima.rob(resex~1, p=2, sd=1, sfreq=12)` |
| `ar3.R` | Table 8.1 | `arima.rob(ar3~1, p=3)` |
| `identAR2.R` | 8.3 | `arima.rob(y~1, auto.ar=TRUE)` |
| `identMA1.R` | 8.4 | `arima.rob(y~1, auto.ar=TRUE)` |
| `MA1-AO.R` | 8.5 | `arima.rob(mac~1, q=1)` |
| `ar1.R` | 8.6 (fig) | simulation/plot only |

`arima.rob` is **deterministic** given its input series; the scripts seed `arima.sim` upstream. All numeric claims are checked **strict-tier** (`atol=0, rtol=0`) against direct R.

In [ ]:
import os, sys, pathlib


import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe for CI execution
import matplotlib.pyplot as plt
import robstattm_py as rpm
from robstattm_py import set_seed
from robstattm_py._r import r as _r

ro = _r()
ro.r("suppressMessages(library(RobStatTM))")
ro.r("suppressMessages(library(robustarima))")
FIG_DIR = pathlib.Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print(f"robstattm_py {rpm.__version__}")

## RESEX — robust seasonal ARIMA (Example 8.6, Figs 8.12–8.13, Table 8.5)

In [ ]:
resex = rpm.datasets.resex()["resex"].to_numpy()
fit = rpm.arima_rob(y=resex, p=2, sd=1, sfreq=12)
print("AR coefficients:", dict(zip(fit.ar_names, np.round(fit.ar, 6))))
print("mean (regcoef):  ", np.round(fit.regcoef, 6))
intercept = fit.regcoef * (1 - fit.ar.sum())
print("intercept:       ", np.round(intercept, 6))

# strict-tier check vs direct R
ro.r("data(resex, package='RobStatTM'); rref <- arima.rob(resex~1, p=2, sd=1, sfreq=12)")
assert np.array_equal(fit.ar, np.asarray(ro.r("as.numeric(rref$model$ar)"), float))
assert np.array_equal(fit.regcoef, np.asarray(ro.r("as.numeric(rref$regcoef)"), float))
assert np.array_equal(fit.y_robust, np.asarray(ro.r("as.numeric(rref$y.robust)"), float))
print("strict-tier vs R: OK")

In [ ]:
# Figure 8.12 analogue: observed series + robustly cleaned series
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, 90), resex, "-", color="0.5", label="RESEX")
ax.plot(np.arange(1, 90), fit.y_robust, "o", ms=3, label="y.robust")
ax.set_xlabel("index"); ax.set_ylabel("RESEX"); ax.legend()
ax.set_title("RESEX — observed vs robustly cleaned")
fig.savefig(FIG_DIR / "ch8_resex.png", dpi=110, bbox_inches="tight"); plt.close(fig)
# Figure 8.13 analogue: sorted |innovations| (TAU) vs LS
innov_tau = np.sort(np.abs(fit.innov[14:89]))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot((np.arange(1, 73)-0.5)/72, innov_tau[:72], "-", label="TAU")
ax.set_xlabel("probability"); ax.set_ylabel("quantiles"); ax.legend()
ax.set_title("RESEX — sorted |innovations| (TAU)")
fig.savefig(FIG_DIR / "ch8_resex_innov.png", dpi=110, bbox_inches="tight"); plt.close(fig)
print("figures saved")

## AR(3) — robust vs LS/MM comparison (Table 8.1)

Simulated AR(3) (`set.seed(600)`), true φ = (4/3, −5/6, 1/6). We reproduce the seeded series in R, then fit `arima.rob(., p=3)` and compare to `lm` / `lmrobdetMM` (already wrapped).

In [ ]:
ro.r("set.seed(600); n.innov<-300; n<-200; phi<-c(4/3,-5/6,1/6); innov<-rnorm(n.innov); ar3<-arima.sim(model=list(ar=phi), n, innov=innov, n.start=n.innov-n)")
ar3 = np.asarray(ro.r("as.numeric(ar3)"), float)
fit3 = rpm.arima_rob(y=ar3, p=3)
ro.r("rref3 <- arima.rob(ar3~1, p=3)")
assert np.array_equal(fit3.ar, np.asarray(ro.r("as.numeric(rref3$model$ar)"), float))
print("AR(3) tau coefficients:", np.round(fit3.ar, 5), "  (true:", [round(4/3,3), round(-5/6,3), round(1/6,3)], ")")
# MM comparator (already wrapped): regress ar3[4:200] on its lags
set_seed(1)
X = np.column_stack([ar3[2:199], ar3[1:198], ar3[0:197]])
mm = rpm.lmrobdet_mm("y ~ x1 + x2 + x3", data=pd.DataFrame({"y": ar3[3:200], "x1": X[:,0], "x2": X[:,1], "x3": X[:,2]}))
print("MM slope coefficients:", np.round(mm.coefficients[1:], 5))

## Automatic AR identification (Examples 8.3 / 8.4)

`arima.rob(y~1, auto.ar=TRUE)` selects the AR order on an outlier-contaminated AR(2) (`identAR2`) and MA(1) (`identMA1`) series. A benign non-convergence warning may appear; the numbers still match R.

In [ ]:
import warnings
ro.r("set.seed(700); n.innov<-300; n<-200; phi<-c(4/3,-5/6); innov<-rnorm(n.innov); x<-arima.sim(model=list(ar=phi), n, innov=innov, n.start=n.innov-n); ao<-ifelse(runif(n)>.1,0,rnorm(n,4,1)); ao<-sign(runif(n,-1,1))*ao; y<-x+ao")
y2 = np.asarray(ro.r("as.numeric(y)"), float)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fa = rpm.arima_rob(y=y2, auto_ar=True)
ro.r("rrefA <- suppressWarnings(arima.rob(y~1, auto.ar=TRUE))")
assert np.array_equal(fa.ar, np.asarray(ro.r("as.numeric(rrefA$model$ar)"), float))
print(f"identAR2: selected AR order = {fa.ar.shape[0]}")
print("AR coefficients:", np.round(fa.ar, 4))

In [ ]:
ro.r("set.seed(600); n.innov<-300; n<-200; theta<-0.8; innov<-rnorm(n.innov); x<-arima.sim(model=list(ma=theta), n, innov=innov, n.start=n.innov-n); ao<-ifelse(runif(n)>.1,0,rnorm(n,6,1)); ao<-sign(runif(n,-1,1))*ao; y<-x+ao")
ym = np.asarray(ro.r("as.numeric(y)"), float)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fm = rpm.arima_rob(y=ym, auto_ar=True)
ro.r("rrefM <- suppressWarnings(arima.rob(y~1, auto.ar=TRUE))")
assert np.array_equal(fm.ar, np.asarray(ro.r("as.numeric(rrefM$model$ar)"), float))
print(f"identMA1: selected AR order = {fm.ar.shape[0]}")

## MA(1) with additive outliers (Example 8.5, Fig 8.11, Table 8.4)

In [ ]:
ro.r("set.seed(200); n.innov<-300; n<-200; theta<--0.8; innov<-rnorm(n.innov); ma1<-arima.sim(model=list(ma=theta), n=n, innov=innov, n.start=n.innov-n); mac<-ma1; mac[20*(1:10)]<-ma1[20*(1:10)]+4")
mac = np.asarray(ro.r("as.numeric(mac)"), float)
fma = rpm.arima_rob(y=mac, q=1)
ro.r("rrefMA <- arima.rob(mac~1, q=1)")
assert np.array_equal(fma.ma, np.asarray(ro.r("as.numeric(rrefMA$model$ma)"), float))
assert np.array_equal(fma.y_robust, np.asarray(ro.r("as.numeric(rrefMA$y.robust)"), float))
print("MA(1) coefficient:", dict(zip(fma.ma_names, np.round(fma.ma, 6))))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, 201), mac, "-", color="0.5", label="series (mac)")
ax.plot(np.arange(1, 201), fma.y_robust, "-", lw=1.2, label="y.robust")
ax.plot(np.arange(20, 201, 20), mac[np.arange(19, 200, 20)], "o", ms=4)
ax.set_xlabel("index"); ax.set_ylabel("series"); ax.legend()
ax.set_title("MA(1) with additive outliers — robust filtering (Fig 8.11)")
fig.savefig(FIG_DIR / "ch8_ma1ao.png", dpi=110, bbox_inches="tight"); plt.close(fig)
print("figure saved")

## AR(1) with AO and IO — simulation only (Fig 8.6)

`ar1.R` loads `robustarima` but only *simulates* and plots — no `arima.rob` fit. We reproduce the three-panel figure.

In [ ]:
ro.r("set.seed(1000); n.innov<-200; n<-100; phi<-0.9; innov<-rnorm(n.innov); x<-arima.sim(model=list(ar=phi), n, innov=innov, n.start=n.innov-n); ao<-rep(0,n); tt<-seq(10,100,10); ao[tt]<-4; xAO<-x+ao; xIO<-x; xIO[50]<-phi*xIO[49]+10; u<-rnorm(50); for (i in 51:100) xIO[i]<-phi*xIO[i-1]+u[i-50]")
x = np.asarray(ro.r("as.numeric(x)"), float)
xAO = np.asarray(ro.r("as.numeric(xAO)"), float)
xIO = np.asarray(ro.r("as.numeric(xIO)"), float)
tt = np.arange(10, 101, 10)
fig, axes = plt.subplots(3, 1, figsize=(7, 8), sharex=True)
axes[0].plot(x); axes[0].set_title("Gaussian AR(1), no outliers")
axes[1].plot(xAO); axes[1].plot(tt, xAO[tt-1], "o"); axes[1].set_title("AR(1) with 10% additive outliers")
axes[2].plot(xIO); axes[2].plot(50, xIO[49], "o"); axes[2].set_title("AR(1) with one innovation outlier")
fig.tight_layout(); fig.savefig(FIG_DIR / "ch8_ar1.png", dpi=110, bbox_inches="tight"); plt.close(fig)
print("Fig 8.6 saved — all six Chapter-8 scripts reproduced.")